Ranking US states and territory by their 2010 population

In [2]:
import pandas as pd
import numpy as np

class Display:
    """
    Display multiple objects side-by-side in a Jupyter notebook

    Class creates HTML representation for each object passed to it. 
    It is useful for comparing multiple Pandas DataFrames or objects that support HTML display.
    """

    # Class HTML template shared between all Display objects/instances
    # {0}: object label
    # {1}: object's HTML representation
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""


    def __init__(self, **objects):
        """
        Store named objects to display
        
        Args: 
            **objects: Named objects to display.
            Keyword name is used as label, and value is the object to display.
        """

        # Store passed objects into a dictionary
        # Store as {label: object}
        self.objects = objects


    def _repr_html_(self): # repr_html special method that displays objects as HTML in notebooks
        """
        Return HTML representation of the stored objects.
        
        Returns:
            str: HTML string showing each object side-by-side
        """

        # Build one HTML block per object
        return '\n'.join( # join HTML blocks with \n
            self.template.format(
                name.capitalize(),  # Object label (capitalized)
                obj._repr_html_()   # Object's HTML display
            )
            for name, obj in self.objects.items() # Iterate over stored objectss
        )


    def __repr__(self): # fallback text representation, used in non-HTML environments
        """
        Return plain-text representation of the stored objects.
        
        Returns:
            str: Text showing each object's name and regular representation
        """

        # __str__ is conceptually like Java's toString() method override
        # __repr__ is a more "precise" representation of the object compared to __str__
        # Build one plain-text block per object instead of HTML
        return '\n\n'.join(
            name + '\n' + repr(obj)
            for name, obj in self.objects.items()
        )

In [3]:
# Load and display datasets
pop = pd.read_csv('data/state-population.csv') #state, ages, year, population
areas = pd.read_csv('data/state-areas.csv') # state, area
abbrevs = pd.read_csv('data/state-abbrevs.csv') # state, abbreviations

Display(
    population=pop.head(),
    areas=areas.head(),
    abbrevs=abbrevs.head()
)

population
  state/region     ages  year  population
0           AL  under18  2012   1117489.0
1           AL    total  2012   4817528.0
2           AL  under18  2010   1130966.0
3           AL    total  2010   4785570.0
4           AL  under18  2011   1125763.0

areas
        state  area (sq. mi)
0     Alabama          52423
1      Alaska         656425
2     Arizona         114006
3    Arkansas          53182
4  California         163707

abbrevs
        state abbreviation
0     Alabama           AL
1      Alaska           AK
2     Arizona           AZ
3    Arkansas           AR
4  California           CA

In [4]:
# Merge pop and areas
merged = pd.merge(
    pop, # left dataframe
    abbrevs, # right dataframe
    how='outer', # outer join, keep all rows
    left_on='state/region', # left (key) column to match/link on
    right_on='abbreviation', # right (key) column to matc/link on
).sort_values('state/region').reset_index(drop=True) # sort data, reset index, drop/dispose old index

merged = merged.drop('abbreviation', axis=1) # drop duplicate info
merged.head()

,state/region,ages,year,population,state
0,AK,total,1990,553290.0,Alaska
1,AK,total,2003,648414.0,Alaska
2,AK,under18,2003,186843.0,Alaska
3,AK,total,2004,659286.0,Alaska
4,AK,under18,2004,186335.0,Alaska


In [5]:
# Check for nulls
merged.isnull().any()

state/region    False
ages            False
year            False
population       True
state            True
dtype: bool

In [6]:
# Look for null values in population column
merged[ # filter rows in merged dataset
    merged['population'].isnull() # boolean mask: true where population is null
    ].head()

,state/region,ages,year,population,state
1897,PR,total,1990,NaN,NaN
1898,PR,total,1991,NaN,NaN
1899,PR,under18,1991,NaN,NaN
1900,PR,total,1993,NaN,NaN
1901,PR,under18,1993,NaN,NaN


In [7]:
# State/region values that don't have corresponding abbreviations
# From rows where state is null, display state/region values column, showing unique values
merged.loc[
    merged['state'].isnull(), # boolean mask: true where state is null
    'state/region' # column to filter
    ].unique() # deduplicate and display unique values

<StringArray>
['PR', 'USA']
Length: 2, dtype: str

In [8]:
# Fill in entries
# Locate rows where state/region is PR or USA, at state column, assign corresponding value
merged.loc[merged['state/region'] == 'PR', 'state'] = 'Puerto Rico'
merged.loc[merged['state/region'] == 'USA', 'state'] = 'United States'

# Check null columns
merged.isnull().any()

state/region    False
ages            False
year            False
population       True
state           False
dtype: bool

In [9]:
# View merged and area datasets
Display(merged=merged.head(), areas=areas.head())

merged
  state/region     ages  year  population   state
0           AK    total  1990    553290.0  Alaska
1           AK    total  2003    648414.0  Alaska
2           AK  under18  2003    186843.0  Alaska
3           AK    total  2004    659286.0  Alaska
4           AK  under18  2004    186335.0  Alaska

areas
        state  area (sq. mi)
0     Alabama          52423
1      Alaska         656425
2     Arizona         114006
3    Arkansas          53182
4  California         163707

In [10]:
# Merge datasets via corresponding state entries and left join
final = pd.merge(merged, areas, on='state', how='left')
final.head()

,state/region,ages,year,population,state,area (sq. mi)
0,AK,total,1990,553290.0,Alaska,656425.0
1,AK,total,2003,648414.0,Alaska,656425.0
2,AK,under18,2003,186843.0,Alaska,656425.0
3,AK,total,2004,659286.0,Alaska,656425.0
4,AK,under18,2004,186335.0,Alaska,656425.0


In [11]:
# Check for nulls in final
final.isnull().any()

state/region     False
ages             False
year             False
population        True
state            False
area (sq. mi)     True
dtype: bool

In [12]:
# Check for the states with null area
final.loc[final['area (sq. mi)'].isnull(), 'state'].unique()

<StringArray>
['United States']
Length: 1, dtype: str

In [13]:
# Since area of United States is null and not relevant, drop it
final.dropna(inplace=True)
final.head()

,state/region,ages,year,population,state,area (sq. mi)
0,AK,total,1990,553290.0,Alaska,656425.0
1,AK,total,2003,648414.0,Alaska,656425.0
2,AK,under18,2003,186843.0,Alaska,656425.0
3,AK,total,2004,659286.0,Alaska,656425.0
4,AK,under18,2004,186335.0,Alaska,656425.0


In [14]:
# Get subset of final with year 2010 and total population
data2010 = final.query("year == 2010 & ages == 'total'")

# Loc method that gets same subset, query is cleaner and more readable
data2010 = final.loc[(final['year'] == 2010) & (final['ages'] == 'total')]

data2010.head()

,state/region,ages,year,population,state,area (sq. mi)
17,AK,total,2010,713868.0,Alaska,656425.0
75,AL,total,2010,4785570.0,Alabama,52423.0
115,AR,total,2010,2922280.0,Arkansas,53182.0
174,AZ,total,2010,6408790.0,Arizona,114006.0
222,CA,total,2010,37333601.0,California,163707.0


In [15]:
# Set index to state
data2010_final = data2010.set_index('state')

# Compute pop density by dividing population by corresponding area
density = data2010_final['population'] / data2010_final['area (sq. mi)']

density.sort_values(ascending=False, inplace=True) # Sort in place by descending order

display(density.head())
display(density.tail())

state
District of Columbia    8898.897059
Puerto Rico             1058.665149
New Jersey              1009.253268
Rhode Island             681.339159
Connecticut              645.600649
dtype: float64

state
South Dakota    10.583512
North Dakota     9.537565
Montana          6.736171
Wyoming          5.768079
Alaska           1.087509
dtype: float64

We processed and merged 3 datasets as well as created feature (density) in order to come up with the conclusion of the highest density states.